# 03. Integrated $L^p$ losses under incomplete sampling

Status: earlier three-seed clustered pilot complete. Revised uniform-versus-clustered study has completed seed 0 for GRU and parameter-matched Linear Neural CDE; seeds 1 and 2 remain. Missingness results form a separate pipeline check.


## 1. Open learning claims

Notebook 01 establishes that unweighted target averages measure residual against empirical sampling measure, while quadrature measures residual against elapsed time. One core learning claim remains open:

1. nonuniform target coverage increases the difference between predictors fitted with unweighted squared error and elapsed-time weighted $J_2$, across conventional and continuous-time architectures.

Each paired run shares paths, observations, model, initial parameters, batch order, optimiser, and training budget. Training objective is the varying factor. Exponent-specific robustness experiments are deferred until a downstream task requires them.


## 2. Generated paths and sampled data

Let the fine reference grid be

$$
t_j=\frac{j}{M},\qquad j=0,\ldots,M,\qquad M=512.
$$

For each training example $i=1,\ldots,n_{\mathrm{train}}$, independently generate a univariate Ornstein–Uhlenbeck path $x^{(i)}:[0,1]\to\mathbb R$. Path index $(i)$ is a parenthesised superscript. Fine grid index $j$ and observation index $r$ are subscripts. Write $x^{(i)}_j:=x^{(i)}(t_j)$. Euler–Maruyama gives

$$
x^{(i)}_{j+1}
=x^{(i)}_j-\theta x^{(i)}_j\Delta t
+\sigma\sqrt{\Delta t}\,\varepsilon^{(i)}_j,\qquad
\varepsilon^{(i)}_j\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),
$$

with $\theta=2$, $\sigma=0.5$, $x^{(i)}_0=1$, and $\Delta t=1/M$. Stored vector $(x^{(i)}_0,\ldots,x^{(i)}_M)$ is fine grid ground truth. Training uses its sampled context and target values.

For path $i$, choose two sorted, disjoint index sets:

$$
C^{(i)}=\{c^{(i)}_1<\cdots<c^{(i)}_C\},\qquad
T^{(i)}=\{q^{(i)}_1<\cdots<q^{(i)}_Q\},\qquad C^{(i)}\cap T^{(i)}=\varnothing,
$$

where $C=Q=64$ and $\{0,M\}\subset T^{(i)}$. Define context and target times for path $i$ by

$$
\tau^{(i)}_r:=t_{c^{(i)}_r},\qquad r=1,\ldots,C,
\qquad\text{and}\qquad
s^{(i)}_r:=t_{q^{(i)}_r},\qquad r=1,\ldots,Q.
$$

The context supplied to the model is

$$
\mathcal O^{(i)}=\Big((\tau^{(i)}_r,x^{(i)}(\tau^{(i)}_r),m^{(i)}_r)\Big)_{r=1}^{C},
$$

and target value is $y^{(i)}_r:=x^{(i)}(s^{(i)}_r)$ for $r=1,\ldots,Q$. Training targets are pairs $(s^{(i)}_r,y^{(i)}_r)$. Clean experiment has $m^{(i)}_r=1$. Disjoint index sets prevent direct copying.


## 3. Target sampling mechanisms

For target times $s^{(i)}_1<\cdots<s^{(i)}_Q$, define empirical sampling measure and elapsed time measure

$$
\mu_Q^{(i)}=\frac1Q\sum_{r=1}^Q\delta_{s^{(i)}_r},
\qquad
\lambda(dt)=\frac{dt}{H^{(i)}},
\qquad H^{(i)}=s^{(i)}_Q-s^{(i)}_1.
$$

Uniform random selection has uniform expected time coverage. Clustered selection uses the same target count with nonuniform expected coverage. Context sampling remains uniform and fixed across target mechanisms, so model input does not change with target mechanism.

| target mechanism | controlled feature |
|---|---|
| uniform random selection | irregular gaps with uniform expected coverage |
| clustered selection | nonuniform time coverage at fixed sample count |

Informative sampling, where observation times depend on path values, is deferred because it requires an observation process model (Weaver, Xiao and Lu, 2023).

Pilot used the clustered law

$$
p_\beta(j)
=\frac{\exp\!\left(\beta(t_j-\tfrac12)\right)}
       {\sum_{k\in\mathcal E}\exp\!\left(\beta(t_k-\tfrac12)\right)}
$$

with $\beta=3$ for context and target selection. Parameter $\beta$ labels this generator and does not characterize general irregularity. Continuous $\beta$ sweep is removed from core design. Time based weighting under clumped observations has direct precedent in Rimoldini (2014).


## 4. GRU input and output

For a batch of $B$ paths, the model receives arrays with shapes

$$
t_{\mathrm{ctx}}\in\mathbb R^{B\times C},\quad
x_{\mathrm{ctx}},m_{\mathrm{ctx}}\in\mathbb R^{B\times C\times1},\quad
t_{\mathrm{query}}\in\mathbb R^{B\times Q}.
$$

For context step $r$, define the time gap $\Delta\tau^{(i)}_1=0$ and $\Delta\tau^{(i)}_r=\tau^{(i)}_r-\tau^{(i)}_{r-1}$ for $r>1$. The encoder input is

$$
v^{(i)}_r=\big(\tau^{(i)}_r,\Delta\tau^{(i)}_r,x^{(i)}(\tau^{(i)}_r),m^{(i)}_r\big)\in\mathbb R^4.
$$

A two layer GRU applies its recurrent update $h^{(i)}_r=\operatorname{GRU}_\theta(v^{(i)}_r,h^{(i)}_{r-1})$ and retains final hidden vector $h^{(i)}_C\in\mathbb R^{64}$. Thus one vector summarises all 64 context observations for path $i$.

A query time $s$ is represented by eight Fourier frequency pairs:

$$
\gamma(s)=\big(s,\sin(2^0\pi s),\cos(2^0\pi s),\ldots,\sin(2^7\pi s),\cos(2^7\pi s)\big).
$$

Decoder is a tanh network with two hidden layers of width 128. Its scalar prediction is

$$
\widehat x^{(i)}_\theta(s)=D_\theta\big(h^{(i)}_C,\gamma(s)\big).
$$

Decoder combines the path summary with each query time representation and returns an array in $\mathbb R^{B\times Q\times1}$. Training updates recurrent encoder and decoder jointly.


## 5. Integrated $L^p$ training objectives

For error $e^{(i)}_r=\widehat x^{(i)}_\theta(s^{(i)}_r)-y^{(i)}_r$, pointwise MSE is

$$
L_{\mathrm{MSE}}(\theta)
=\frac1B\sum_{i=1}^B\frac1Q\sum_{r=1}^Q\left|e^{(i)}_r\right|^2.
$$

It assigns weight $1/Q$ to every target, irrespective of the gaps between target times.

For sorted target times $s^{(i)}_1<\cdots<s^{(i)}_Q$, define normalised trapezoid weights

$$
w^{(i)}_1=\frac{s^{(i)}_2-s^{(i)}_1}{2H^{(i)}},\qquad
w^{(i)}_r=\frac{s^{(i)}_{r+1}-s^{(i)}_{r-1}}{2H^{(i)}}\ (1<r<Q),\qquad
w^{(i)}_Q=\frac{s^{(i)}_Q-s^{(i)}_{Q-1}}{2H^{(i)}},
$$

where $H^{(i)}=s^{(i)}_Q-s^{(i)}_1=1$ and $\sum_r w^{(i)}_r=1$. Weight concentration is summarised by

$$
n_{\mathrm{eff}}^{(i)}
=\frac{1}{\sum_{r=1}^Q\left(w^{(i)}_r\right)^2}.
$$

Equal weights give the maximum $n_{\mathrm{eff}}=Q$; weight concentration reduces it. A regular trapezoid grid lies slightly below $Q$ because its two endpoint weights are halved. For finite $p\geq1$, define

$$
J_p(\theta)
=\frac1B\sum_{i=1}^B\sum_{r=1}^Qw^{(i)}_r\left|e^{(i)}_r\right|^p
\approx\frac1B\sum_{i=1}^B\left\|e^{(i)}_\theta\right\|_{L^p}^p.
$$

Implementation omits $p$th root. Thus $J_1$ is integrated absolute error, $J_2$ is integrated squared error, and $J_4$ is integrated fourth power. For one path, root and power have same minimiser. Averaging across paths before or after taking root can change a fitted model.

Supremum loss is

$$
J_\infty(\theta)
=\frac1B\sum_{i=1}^B\max_{1\leq r\leq Q}\left|e^{(i)}_r\right|.
$$

It is independent of quadrature weights and serves as a diagnostic because a sampled maximum can miss errors inside long gaps.


## 6. Sampling measure and exponent

Define residual path $e^{(i)}_\theta(t):=\widehat x^{(i)}_\theta(t)-x^{(i)}(t)$. Two independent choices define an integrated loss: measure on time and exponent on residual magnitude.

If empirical sampling measures converge to density $\rho$, unweighted $p$ power estimates

$$
\int_0^1 \left|e^{(i)}_\theta(t)\right|^p\rho(t)\,dt,
$$

while quadrature approximates

$$
\int_0^1 \left|e^{(i)}_\theta(t)\right|^p\,dt.
$$

Measure choice changes time emphasis. Exponent choice changes statistical target and tail emphasis. For $p>1$, unrestricted pointwise optimum $a_p(t)$ satisfies

$$
\mathbb E\!\left[
|X(t)-a_p(t)|^{p-2}(a_p(t)-X(t))
\mid\mathcal O
\right]=0.
$$

| exponent | pointwise target and emphasis |
|---:|---|
| $1$ | conditional median; linear outlier penalty |
| $2$ | conditional mean; quadratic penalty and Hilbert geometry |
| $4$ | centre weighted toward tail errors; strong peak penalty |
| $\infty$ | worst sampled time |

Absolute and squared losses elicit median and mean respectively (Gneiting, 2011). $L^2$ dominates functional data analysis because inner product structure supports bases, projection, and functional regression (Ramsay and Silverman, 2005).

Conditional Ornstein–Uhlenbeck distribution is Gaussian and symmetric. Its mean and median coincide, so clean OU data are a control where $p=1$ and $p=2$ have same unrestricted pointwise optimum. Exponent study needs contamination or asymmetric errors to expose robustness, and local pulses to expose peak emphasis.

Every raw $L^p$ objective compares aligned residual magnitudes. Measure preserving rearrangement of time leaves its value unchanged. Temporal alignment losses such as soft DTW address correspondence (Cuturi and Blondel, 2017); signatures address ordered path interactions.


## 7. Paired training and evaluation algorithm

Planned studies use the following protocol for each seed $a$, data condition $d$, and objective $J_p$ under comparison.

1. Set the NumPy and PyTorch random seeds to $a$.
2. Generate fixed train, validation, and test splits containing 512, 128, and 256 independent base paths. Store fine grid truth for every path.
3. Generate one fixed context set for each path. Generate target sets according to condition $d$. Within $(a,d)$, paths, context, targets, and minibatch order are shared across objectives.
4. Initialise the same model parameters $\theta_0$ for every paired loss comparison. Create Adam with learning rate $10^{-3}$.
5. For epochs $k=1,\ldots,200$, draw a shared random permutation, divide it into batches of $B=64$ paths, predict at all target times, compute the chosen objective, back propagate, and take one Adam step.
6. Evaluate the final model once on 256 held out test paths.
7. On all 513 fine grid times, report $J_1$, $J_2$, $J_4$, and mean pathwise $J_\infty$ for every fitted model.
8. Compare objectives within seed, architecture, and target mechanism, then summarise paired differences across seeds.

The completed pilot predates this protocol. It uses 512 training and 128 validation paths, three seeds, one clustered mechanism, and no independent test split. Its values remain exploratory validation estimates.


## 8. Completed pilot configuration

| component | value |
|---|---|
| path process | univariate Ornstein–Uhlenbeck, $\theta=2$, $\sigma=0.5$, and $x^{(i)}(0)=1$ for every path $i$ |
| fine grid | 513 points on $[0,1]$ |
| data | 512 training paths, 128 validation paths |
| samples per path | 64 context, 64 disjoint targets including endpoints |
| sampling | clustered, $\beta=3$; both context and target affected |
| corruption | none |
| model | two layer GRU, hidden width 64; tanh decoder width 128; 8 Fourier frequencies |
| optimisation | Adam, learning rate $10^{-3}$, batch 64, 200 epochs |
| paired seeds | 0, 1, 2 |


In [ ]:
import json
from pathlib import Path
from statistics import mean

root = Path('../results/runs/loss_comparison')
pilot = {}
for path in sorted(root.glob('*_seed*/final.json')):
    loss_name, seed = path.parent.name.rsplit('_seed', 1)
    pilot[(loss_name, int(seed))] = json.loads(path.read_text())

metrics = ['fine_mse', 'fine_integral_l2', 'fine_integral_l1', 'fine_integral_linf']
summary = {
    loss: {metric: mean(pilot[(loss, seed)][metric] for seed in range(3)) for metric in metrics}
    for loss in ['mse', 'integral_l2']
}
paired_primary = [
    {
        'seed': seed,
        'mse_trained': pilot[('mse', seed)]['fine_integral_l2'],
        'weighted_l2_trained': pilot[('integral_l2', seed)]['fine_integral_l2'],
        'relative_improvement_percent': 100 * (
            pilot[('mse', seed)]['fine_integral_l2']
            - pilot[('integral_l2', seed)]['fine_integral_l2']
        ) / pilot[('mse', seed)]['fine_integral_l2'],
    }
    for seed in range(3)
]
summary, paired_primary


## 9. Pilot results

Mean fine grid validation metrics across three paired seeds:

| training objective | MSE | squared $L^2$ | $L^1$ | mean pathwise $L^\infty$ |
|---|---:|---:|---:|---:|
| pointwise MSE | 0.008476 | 0.008480 | 0.071919 | 0.250602 |
| time weighted squared $L^2$ | 0.008135 | 0.008133 | 0.071366 | 0.245308 |

Primary fine grid squared $L^2$ results by seed:

| seed | MSE trained | weighted $L^2$ trained | relative change |
|---:|---:|---:|---:|
| 0 | 0.008307 | 0.007996 | 3.7% improvement |
| 1 | 0.008372 | 0.008459 | 1.0% deterioration |
| 2 | 0.008763 | 0.007944 | 9.3% improvement |

Mean primary score decreases by 4.1% under weighted training. Two of three seeds improve. Stored runs yield different fitted solutions under the two objectives. The sampling mechanism responsible for the difference and any general performance ordering remain open.


## 10. Core architecture and sampling study

Hold uniformly sampled context and $Q=64$ targets fixed. Compare uniform random target selection with clustered target selection. For each mechanism, fit paired models with unweighted squared error and time weighted $J_2$. Run both GRU and parameter matched Linear Neural CDE architectures over three seeds. GRU has 65,537 parameters and Neural CDE has 65,121.

For seed $a$ and mechanism $m$, define

$$
\Delta_{a,m}
=R_{a,m}(J_2\text{ training})
-R_{a,m}(\text{unweighted squared training}),
$$

where $R$ is fine grid $J_2$ on 256 held out test paths. Negative values favour time weighting. Uniform target selection is the coverage control. Clustered target selection tests the sampling-measure mechanism. A larger weighted-loss effect under clustering, reproduced across architectures and seeds, supports that mechanism.

### 10.1 Seed 0 pipeline result

| architecture | target mechanism | MSE trained test $J_2$ | weighted-$J_2$ trained test $J_2$ | relative decrease |
|---|---|---:|---:|---:|
| GRU | uniform | 0.006645 | 0.006373 | 4.1% |
| GRU | clustered | 0.007285 | 0.006594 | 9.5% |
| Linear Neural CDE | uniform | 0.007768 | 0.007459 | 4.0% |
| Linear Neural CDE | clustered | 0.009395 | 0.007865 | 16.3% |

Weighted training improves all four seed 0 comparisons, with larger relative change under clustered targets for both architectures. One seed verifies the revised pipeline and supplies a directional result. Seeds 1 and 2 are required for a stable claim.

### 10.2 Deferred exponent extensions

$J_1$ and $J_4$ remain implemented and reported. Separate contamination and local-peak experiments are deferred because they answer application-specific robustness questions rather than the current sampling-measure claim. Reintroduce an exponent study only when data or a downstream task supplies that requirement.


## 11. Separate context missingness study

Target sampling changes the measure used by the training loss. Context missingness changes the information supplied to the model. It is therefore a separate input robustness study. For each clean context value draw one fixed variable $U^{(i)}_r\sim\operatorname{Uniform}(0,1)$ and, at rate $\rho$, set

$$
m^{(i)}_r(\rho)=\mathbf 1\{U^{(i)}_r\geq\rho\},\qquad
\widetilde x^{(i)}_r(\rho)=m^{(i)}_r(\rho)x^{(i)}(\tau^{(i)}_r).
$$

For path $i$, model receives zero filled values $\widetilde x^{(i)}_r(\rho)$ and masks $m^{(i)}_r(\rho)$. Reusing $U^{(i)}_r$ across rates makes masks nested: every value missing at 10% remains missing at 30% and 50%. Query targets and fine grid ground truth stay fixed.

Robustness experiment first evaluates fixed clean trained models at $\rho\in\{0,0.1,0.3,0.5\}$ across paired seeds. A separate experiment trains with missingness augmentation.


In [ ]:
missingness_path = Path('../results/runs/missingness_mse_seed0/robustness.json')
missingness_check = json.loads(missingness_path.read_text())['missingness']
{
    rate: {
        'fine_integral_l2': values['fine_integral_l2'],
        'fine_integral_linf': values['fine_integral_linf'],
    }
    for rate, values in missingness_check.items()
}


### Exploratory pipeline check

One clean trained, MSE trained seed produced:

| missing context values | fine grid squared $L^2$ | mean pathwise $L^\infty$ |
|---:|---:|---:|
| 0% | 0.008307 | 0.244691 |
| 10% | 0.009616 | 0.260945 |
| 30% | 0.017053 | 0.319491 |
| 50% | 0.027741 | 0.375408 |

One seed verifies masking and evaluation plumbing. Stable degradation estimates and loss or architecture comparisons require paired seeds.


## 12. Reproducibility map

- Path generation and index sampling: `src/pathloss/paths.py` and `src/pathloss/datasets.py`
- GRU query model: `src/pathloss/models.py`
- MSE, trapezoid weights, and squared $L^2$: `src/pathloss/losses.py`
- Training and evaluation loop: `src/pathloss/train.py`
- Completed paired outputs: `results/runs/loss_comparison/`
- Exploratory missingness output: `results/runs/missingness_mse_seed0/robustness.json`

Core study runner: `scripts/run_integral_study.py`; configuration: `configs/integral_core_study.yaml`; ARC array: `scripts/arc/submit_integral_core_array.slurm`; seed 0 outputs: `results/runs/integral_core/`. Seeds 1 and 2 remain.
